In [25]:
# Figures for "Ray Theory I: Snell’s Law and the Ray Parameter"
# ------------------------------------------------------------
# These figures are *original* schematic/teaching figures inspired by standard ray theory geometry.
# They are NOT reproductions of Shearer figures; they are simplified conceptual diagrams you can
# modify freely for your lecture/JupyterBook.
#
# Output: PNG files written to ./figures/
#
# Usage:
#   1) Run this cell (or save as make_ray_figs_part1.py and run it).
#   2) Include in your JupyterBook with:
#        ![](figures/fig1_plane_wave_incident.png)
#        ![](figures/fig2_snell_timing_continuity.png)
#        ![](figures/fig3_curved_rays_turning.png)
#        ![](figures/fig4_traveltime_tangent_p.png)

from __future__ import annotations

import os
import numpy as np
import matplotlib.pyplot as plt


# -------------------------
# Global style (minimal)
# -------------------------
plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 200,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

OUTDIR = "figures"
os.makedirs(OUTDIR, exist_ok=True)


def _save(fig: plt.Figure, fname: str) -> None:
    path = os.path.join(OUTDIR, fname)
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {path}")



def _save_pdf(fig: plt.Figure, fname: str) -> None:
    path = os.path.join(OUTDIR, fname)
    fig.tight_layout()
    fig.savefig(path, format="pdf")
    plt.close(fig)
    print(f"Saved: {path}")



In [26]:

# ============================================================
# Figure 1: Plane wave incident on a horizontal interface
# ============================================================
def fig1_plane_wave_incident(
    theta_deg: float = 35.0,
    interface_depth: float = 0.0,
    n_wavefronts: int = 7,
    spacing: float = 0.6,
) -> None:
    """
    Schematic: plane wavefronts and a ray incident on a horizontal interface.
    Angle is measured from vertical (seismology convention).
    """
    theta = np.deg2rad(theta_deg)

    # Coordinate system: x horizontal, z vertical (down positive)
    # We'll plot z upward for figure aesthetics (so positive "up"); but label accordingly.
    # Keep geometry consistent: use y = depth with "up" positive in plot.
    # Here: y = -z (so surface is higher)
    fig, ax = plt.subplots(figsize=(7.2, 3.6))

    # Interface line (horizontal)
    ax.axhline(interface_depth, lw=1.5)
    ax.text(3.1, interface_depth + 0.15, "interface", va="bottom")

    # Define a ray line crossing the interface at (x0, y0)
    x0, y0 = 1.6, interface_depth
    # angle from vertical => ray direction unit vector:
    # In x-y plot: y positive up, so down-going ray has negative y component.
    dx = np.sin(theta)
    dy = -np.cos(theta)

    # Ray segment
    t = np.linspace(-2.0, 2.0, 100)
    xr = x0 + t * dx
    yr = y0 + t * dy
    ax.plot(xr, yr, lw=2.0)
    ax.annotate("", xy=(x0 + 1.2 * dx, y0 + 1.2 * dy), xytext=(x0 + 0.6 * dx, y0 + 0.6 * dy),
                arrowprops=dict(arrowstyle="->", lw=2.0))

    # Wavefronts: lines perpendicular to ray direction, spaced along the ray
    # A wavefront normal is along ray direction; wavefront line direction is perpendicular.
    # Perp unit vector:
    px, py = dy, -dx

    # Choose wavefront anchor points along ray above interface
    s_vals = np.arange(-(n_wavefronts // 2), (n_wavefronts // 2) + 1) * spacing
    for s in s_vals:
        xa = x0 + s * dx
        ya = y0 + s * dy
        # wavefront line segment
        L = 2.5
        xw = xa + np.array([-L, L]) * px
        yw = ya + np.array([-L, L]) * py
        ax.plot(xw, yw, lw=1.0)

    # Draw incidence angle arc from vertical at interface
    # Vertical reference line
    ax.plot([x0, x0], [y0, y0 - 1.3], lw=1.2, ls="--")
    # Arc
    arc_t = np.linspace(-np.pi/2, -np.pi/2 + theta, 50)
    r = 0.55
    ax.plot(x0 + r * np.cos(arc_t), y0 + r * np.sin(arc_t), lw=1.2)
    ax.text(x0 + 0.45*np.sin(theta/2), y0 - 0.45*np.cos(theta/2), r"$\theta$", ha="center", va="center")

    # Annotate Δx along interface between two consecutive wavefront intersections
    # Pick two nearby wavefronts and compute their intersection with interface y = y0
    # Solve for parameter alpha in wavefront line: (x,y) = (xa,ya) + alpha*(px,py)
    # Set y = y0 -> alpha = (y0 - ya)/py
    # Choose s = -spacing and s = 0
    def intersect_x(s):
        xa = x0 + s * dx
        ya = y0 + s * dy
        alpha = (y0 - ya) / py
        return xa + alpha * px

    x_int1 = intersect_x(-spacing)
    x_int2 = intersect_x(0.0)
    ax.annotate("", xy=(x_int2, y0), xytext=(x_int1, y0),
                arrowprops=dict(arrowstyle="<->", lw=1.4))
    ax.text((x_int1 + x_int2)/2, y0 + 0.12, r"$\Delta x$", ha="center", va="bottom")

    # Annotate Δs along ray between wavefronts
    ax.annotate("", xy=(x0, y0), xytext=(x0 - spacing*dx, y0 - spacing*dy),
                arrowprops=dict(arrowstyle="<->", lw=1.4))
    ax.text(x0 - 0.55*spacing*dx + 0.08, y0 - 0.55*spacing*dy - 0.1, r"$\Delta s$", ha="left", va="top")

    ax.set_title("Plane wave incident on a horizontal interface (schematic)")
    ax.set_xlabel("x (horizontal)")
    ax.set_ylabel("y (up)")
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-0.3, 4.2)
    ax.set_ylim(-3.0, 1.6)

    _save(fig, "fig1_plane_wave_incident.png")

    _save_pdf(fig, "fig1_plane_wave_incident.pdf")


In [33]:

# ============================================================
# Figure 2: Snell timing continuity across interface
# ============================================================
def fig2_snell_timing_continuity(
    theta1_deg: float = 35.0,
    v1: float = 4.0,
    v2: float = 6.0,
    interface_y: float = 0.0,
    n_wavefronts: int = 5,
) -> None:
    """
    Schematic: transmitted wavefront spacing changes across layers, enforcing Snell's law.
    We show two bundles of wavefronts above and below interface and rays with different angles.
    """
    theta1 = np.deg2rad(theta1_deg)
    # Snell: sin(theta1)/v1 = sin(theta2)/v2 => theta2 = arcsin( (v2/v1)*sin(theta1) )?
    # Careful: seismology convention uses theta from vertical. Snell is u1 sinθ1 = u2 sinθ2.
    # => sinθ2 = (u1/u2) sinθ1 = (v2/v1) sinθ1. With v2>v1, θ2 increases (bends away from vertical).
    sin_theta2 = (v2 / v1) * np.sin(theta1)
    if sin_theta2 >= 1.0:
        # Critical refraction case; clamp for schematic but label
        sin_theta2 = 0.999
    theta2 = np.arcsin(sin_theta2)

    # Wavefront spacings along ray: Δs = v Δt, so spacing proportional to v
    spacing1 = 0.55
    spacing2 = spacing1 * (v2 / v1)

    fig, ax = plt.subplots(figsize=(7.2, 4.2))

    # Add shaded regions to show different media
    ax.fill_between([-0.5, 5.0], [interface_y, interface_y], [1.6, 1.6], 
                    alpha=0.15, color='lightblue', label='Medium 1 (v₁)')
    ax.fill_between([-0.5, 5.0], [-3.4, -3.4], [interface_y, interface_y], 
                    alpha=0.25, color='lightsalmon', label='Medium 2 (v₂)')

    # interface
    ax.axhline(interface_y, lw=1.5, color='black')
    ax.text(4.6, interface_y + 0.12, "interface", ha="right", va="bottom")

    # Anchor point at interface
    x0, y0 = 2.2, interface_y

    # Draw vertical normal line (dashed green) extending through interface
    ax.plot([x0, x0], [-3.4, 1.6], 'g--', linewidth=2, label='Normal', zorder=3)

    # Ray direction above (down-going)
    dx1, dy1 = np.sin(theta1), -np.cos(theta1)
    # below
    dx2, dy2 = np.sin(theta2), -np.cos(theta2)

    # Plot rays above and below with distinct colors
    t1 = np.linspace(-2.0, 0.0, 80)  # above interface (y>0 region)
    ax.plot(x0 + t1*dx1, y0 + t1*dy1, lw=3.0, color='darkblue', zorder=4)
    t2 = np.linspace(0.0, 3.0, 120)
    ax.plot(x0 + t2*dx2, y0 + t2*dy2, lw=3.0, color='darkorange', zorder=4)

    # Wavefronts above (perpendicular to ray) with colormap
    px1, py1 = dy1, -dx1
    s_vals1 = np.arange(-(n_wavefronts//2), (n_wavefronts//2)+1) * spacing1
    # Create colormap for propagation visualization
    cmap = plt.cm.YlOrRd
    # Normalize positions to [0, 1] for colormap (earlier = cooler, later = warmer)
    norm_vals1 = (s_vals1 - s_vals1.min()) / (s_vals1.max() - s_vals1.min())
    for i, s in enumerate(s_vals1):
        xa = x0 + s*dx1
        ya = y0 + s*dy1
        L = 2.7
        color = cmap(norm_vals1[i])
        ax.plot(xa + np.array([-L, L])*px1, ya + np.array([-L, L])*py1, 
                lw=1.5, color=color, alpha=0.7, zorder=2)

    # Wavefronts below with colormap
    px2, py2 = dy2, -dx2
    s_vals2 = np.arange(0, n_wavefronts) * spacing2  # start at interface and go down
    # Continue colormap progression from layer 1
    norm_vals2 = np.linspace(0.5, 1.0, len(s_vals2))  # later times in propagation
    for i, s in enumerate(s_vals2):
        xa = x0 + s*dx2
        ya = y0 + s*dy2
        L = 2.7
        color = cmap(norm_vals2[i])
        ax.plot(xa + np.array([-L, L])*px2, ya + np.array([-L, L])*py2, 
                lw=1.5, color=color, alpha=0.7, zorder=2)

    # Angle θ₁ (incident angle from normal)
    arc1 = np.linspace(-np.pi/2, -np.pi/2 + theta1, 50)
    r = 0.6
    ax.plot(x0 + r*np.cos(arc1), y0 + r*np.sin(arc1), lw=2, color='blue', zorder=5)
    ax.text(x0 + 0.5*np.sin(theta1/2), y0 - 0.5*np.cos(theta1/2), r"$\theta_1$", 
            ha="center", va="center", fontsize=12, color='blue', fontweight='bold', zorder=5)

    # Angle θ₂ (refracted angle from normal)
    arc2 = np.linspace(-np.pi/2, -np.pi/2 + theta2, 50)
    r2 = 0.6
    ax.plot(x0 + r2*np.cos(arc2), y0 + r2*np.sin(arc2), lw=2, color='darkorange', zorder=5)
    ax.text(x0 + 0.5*np.sin(theta2/2), y0 - 0.5*np.cos(theta2/2), r"$\theta_2$", 
            ha="center", va="center", fontsize=12, color='darkorange', fontweight='bold', zorder=5)
    # Add colorbar to show time progression
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=0, vmax=1))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, orientation='vertical', pad=0.02, shrink=0.7)
    cbar.set_label('Time progression', rotation=270, labelpad=15)
    cbar.set_ticks([0, 1])
    cbar.set_ticklabels(['Early', 'Late'])

    ax.set_title("Timing continuity across an interface (Snell's law schematic)")
    ax.set_xlabel("x (horizontal)")
    ax.set_ylabel("y (depth)")
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-0.5, 5.0)
    ax.set_ylim(-3.4, 1.6)

    _save(fig, "fig2_snell_timing_continuity.png")
    _save_pdf(fig, "fig2_snell_timing_continuity.pdf")



In [28]:

# ============================================================
# Figure 3: Curved rays and turning points in v(z) increasing
# ============================================================
def fig3_curved_rays_turning(
    vmax: float = 7.5,
    v0: float = 4.0,
    zmax: float = 60.0,
    n_rays: int = 5,
    x_max: float = 220.0,
) -> None:
    """
    Schematic: multiple rays in a depth-dependent velocity model v(z) increasing with depth.
    We generate stylized raypaths (not numerically exact) using a simple parametric form that
    resembles turning rays and clearly marks turning points.
    """
    fig, ax = plt.subplots(figsize=(7.2, 4.4))

    # Depth axis: z positive downward
    z = np.linspace(0, zmax, 300)
    # Smoothly increasing velocity (for annotation only)
    v = v0 + (vmax - v0) * (1 - np.exp(-z / (0.35*zmax)))

    # Plot a side panel-like velocity curve faintly in background as reference
    # We'll normalize v to x-range to show as backdrop
    x_v = 10 + 40 * (v - v.min()) / (v.max() - v.min())
    ax.plot(x_v, z, lw=1.2, ls="--")
    ax.text(x_v[-1] + 5, z[-1], r"$v(z)$", va="center")

    # Stylized rays: x(s) = A*sin(pi*s), z(s) = zturn*sin(pi*s/2)^2 shape
    s = np.linspace(0, 1, 250)
    # Choose turning depths (deeper for smaller p / steeper)
    z_turns = np.linspace(12, 52, n_rays)
    # Choose ranges (larger for deeper turning)
    x_ranges = np.linspace(70, x_max, n_rays)

    for zt, xr in zip(z_turns, x_ranges):
        x = (xr / 2.0) * (1 - np.cos(np.pi * s))   # 0 -> xr
        zray = zt * np.sin(np.pi * s) ** 2         # 0 -> zt -> 0
        ax.plot(x, zray, lw=2.0)
        # Mark turning point (s=0.5)
        it = np.argmin(np.abs(s - 0.5))
        ax.plot(x[it], zray[it], marker="o", ms=4)
        ax.text(x[it] + 3, zray[it] + 1.0, "turning\npoint", ha="left", va="bottom", fontsize=9)

    # Surface line
    ax.axhline(0, lw=1.5)
    ax.text(3, 1.5, "surface", va="bottom")

    ax.set_title("Rays in a model with velocity increasing with depth (turning rays schematic)")
    ax.set_xlabel("x (horizontal distance km)")
    ax.set_ylabel("z (depth km)")
    ax.set_xlim(-2, x_max + 25)
    ax.set_ylim(zmax + 5, -5)  # invert so depth increases downward (common in seismology figures)

    _save(fig, "fig3_curved_rays_turning.png")
    _save_pdf(fig, "fig3_curved_rays_turning.pdf")

In [29]:

# ============================================================
# Figure 4: Travel-time curve with tangent slope = p
# ============================================================
def fig4_traveltime_tangent_p(
    vmin: float = 3.0,
    vdeep: float = 8.5,
    x_max: float = 300.0,
) -> None:
    """
    Create a smooth, plausible prograde travel-time curve and annotate that slope dT/dX is p.
    This is a conceptual plot (not tied to a specific Earth model), useful for pedagogy.
    """
    fig, ax = plt.subplots(figsize=(7.2, 4.0))

    X = np.linspace(0, x_max, 400)
    # Construct a concave travel-time curve with decreasing slope (faster effective velocity at longer range)
    # T ~ X / v_eff(X) where v_eff increases with range
    v_eff = vmin + (vdeep - vmin) * (1 - np.exp(-X / (0.35*x_max)))
    T = X / v_eff

    ax.plot(X, T, lw=2.2)
    ax.set_title("Travel-time curve and ray parameter $p = dT/dX$")
    ax.set_xlabel("X (distance km)")
    ax.set_ylabel("T (travel time s)")

    # Pick a point and draw tangent
    X0 = 0.62 * x_max
    i0 = np.argmin(np.abs(X - X0))
    T0 = T[i0]
    # numerical derivative
    dT_dX = np.gradient(T, X)[i0]

    # Tangent line
    Xt = np.linspace(X0 - 0.22*x_max, X0 + 0.22*x_max, 50)
    Tt = T0 + dT_dX * (Xt - X0)
    ax.plot(Xt, Tt, lw=1.8, ls="--")

    ax.plot([X0], [T0], marker="o", ms=5)
    ax.text(X0 + 8, T0 + 0.05*(T.max()-T.min()),
            rf"tangent slope $p = dT/dX \approx {dT_dX:.3f}$ (s/km)", ha="left", va="bottom")

    # Small right triangle to visually suggest slope
    dx = 0.07 * x_max
    dy = dT_dX * dx
    ax.plot([X0, X0 + dx], [T0, T0], lw=1.2)
    ax.plot([X0 + dx, X0 + dx], [T0, T0 + dy], lw=1.2)
    ax.plot([X0, X0 + dx], [T0, T0 + dy], lw=1.2)
    ax.text(X0 + dx/2, T0 - 0.04*(T.max()-T.min()), r"$\Delta X$", ha="center", va="top")
    ax.text(X0 + dx + 4, T0 + dy/2, r"$\Delta T$", ha="left", va="center")

    _save(fig, "fig4_traveltime_tangent_p.png")
    _save_pdf(fig, "fig4_traveltime_tangent_p.pdf")




In [32]:
fig1_plane_wave_incident()
fig2_snell_timing_continuity()
fig3_curved_rays_turning()
fig4_traveltime_tangent_p()

Saved: figures/fig1_plane_wave_incident.png
Saved: figures/fig1_plane_wave_incident.pdf
Saved: figures/fig2_snell_timing_continuity.png
Saved: figures/fig2_snell_timing_continuity.pdf
Saved: figures/fig3_curved_rays_turning.png
Saved: figures/fig3_curved_rays_turning.pdf
Saved: figures/fig4_traveltime_tangent_p.png
Saved: figures/fig4_traveltime_tangent_p.pdf
